In [ ]:
"""
Name: Youssef Badawi Sedqi
ID: 4241018


Tic Tac Toe with Minimax Algorithm
===================================
Player (X) vs AI (O) — AI uses the Minimax algorithm and never loses.
Run: python tic_tac_toe.py
"""

import tkinter as tk
from tkinter import font as tkfont
import math
import time


# ─────────────────────────────────────────
#  MINIMAX LOGIC
# ─────────────────────────────────────────

def check_winner(board):
    """Return 'X', 'O', 'draw', or None."""
    wins = [
        [0,1,2],[3,4,5],[6,7,8],   # rows
        [0,3,6],[1,4,7],[2,5,8],   # cols
        [0,4,8],[2,4,6]            # diags
    ]
    for a,b,c in wins:
        if board[a] == board[b] == board[c] and board[a] != '':
            return board[a], (a,b,c)
    if '' not in board:
        return 'draw', None
    return None, None


def minimax(board, is_maximizing, alpha=-math.inf, beta=math.inf, depth=0):
    winner, _ = check_winner(board)
    if winner == 'O':  return 10 - depth
    if winner == 'X':  return depth - 10
    if winner == 'draw': return 0

    if is_maximizing:           # AI = O
        best = -math.inf
        for i in range(9):
            if board[i] == '':
                board[i] = 'O'
                best = max(best, minimax(board, False, alpha, beta, depth+1))
                board[i] = ''
                alpha = max(alpha, best)
                if beta <= alpha: break
        return best
    else:                       # Human = X
        best = math.inf
        for i in range(9):
            if board[i] == '':
                board[i] = 'X'
                best = min(best, minimax(board, True, alpha, beta, depth+1))
                board[i] = ''
                beta = min(beta, best)
                if beta <= alpha: break
        return best


def best_move(board):
    best_val = -math.inf
    move = -1
    for i in range(9):
        if board[i] == '':
            board[i] = 'O'
            val = minimax(board, False)
            board[i] = ''
            if val > best_val:
                best_val = val
                move = i
    return move


# ─────────────────────────────────────────
#  GUI
# ─────────────────────────────────────────

class TicTacToe(tk.Tk):
    # Palette
    BG      = "#0d0d0f"
    PANEL   = "#13131a"
    ACCENT  = "#7c3aed"       # violet
    ACCENT2 = "#f59e0b"       # amber
    TEXT    = "#e8e8f0"
    DIM     = "#3a3a50"
    WIN_HL  = "#4ade80"       # green flash

    def __init__(self):
        super().__init__()
        self.title("TIC · TAC · TOE")
        self.resizable(False, False)
        self.configure(bg=self.BG)

        self.board = [''] * 9
        self.game_over = False
        self.player_turn = True          # True = human (X)
        self.scores = {'X': 0, 'O': 0, 'draw': 0}
        self.buttons = []
        self.win_cells = []

        self._build_fonts()
        self._build_ui()
        self.center_window()

    # ── fonts ──────────────────────────────
    def _build_fonts(self):
        self.f_title  = tkfont.Font(family="Courier New", size=18, weight="bold")
        self.f_mark   = tkfont.Font(family="Courier New", size=44, weight="bold")
        self.f_status = tkfont.Font(family="Courier New", size=11)
        self.f_score  = tkfont.Font(family="Courier New", size=10, weight="bold")
        self.f_btn    = tkfont.Font(family="Courier New", size=9,  weight="bold")

    # ── layout ─────────────────────────────
    def _build_ui(self):
        outer = tk.Frame(self, bg=self.BG, padx=30, pady=24)
        outer.pack()

        # Title
        title = tk.Label(outer, text="TIC · TAC · TOE",
                         font=self.f_title, fg=self.ACCENT, bg=self.BG)
        title.pack(pady=(0,4))

        sub = tk.Label(outer, text="you  ×  vs  AI  ○",
                       font=self.f_status, fg=self.DIM, bg=self.BG)
        sub.pack(pady=(0,16))

        # Score bar
        self.score_frame = tk.Frame(outer, bg=self.PANEL, pady=8)
        self.score_frame.pack(fill="x", pady=(0,16))

        self._score_label('YOU (X)', self.ACCENT, 0)
        tk.Label(self.score_frame, text="·", fg=self.DIM,
                 bg=self.PANEL, font=self.f_score).grid(row=0,column=1,padx=10)
        self._score_label('AI  (O)', self.ACCENT2, 2)

        # Status
        self.status_var = tk.StringVar(value="▶  Your turn  (X)")
        self.status_lbl = tk.Label(outer, textvariable=self.status_var,
                                   font=self.f_status, fg=self.TEXT, bg=self.BG)
        self.status_lbl.pack(pady=(0,14))

        # Grid
        grid_frame = tk.Frame(outer, bg=self.DIM, padx=2, pady=2)
        grid_frame.pack()

        for i in range(9):
            r, c = divmod(i, 3)
            btn = tk.Button(
                grid_frame, text='', font=self.f_mark,
                width=3, height=1,
                bg=self.PANEL, fg=self.TEXT,
                activebackground=self.BG,
                relief="flat", bd=0,
                cursor="hand2",
                command=lambda idx=i: self.human_move(idx)
            )
            btn.grid(row=r, column=c, padx=2, pady=2, ipadx=10, ipady=10)
            self.buttons.append(btn)

        # Scoreboard vars (after buttons)
        self.x_score_var = tk.StringVar(value="0")
        self.o_score_var = tk.StringVar(value="0")
        self._refresh_scores()

        # Restart
        restart_btn = tk.Button(
            outer, text="⟳  NEW GAME",
            font=self.f_btn, fg=self.TEXT, bg=self.ACCENT,
            activebackground="#6d28d9",
            relief="flat", bd=0, padx=18, pady=8,
            cursor="hand2",
            command=self.reset_game
        )
        restart_btn.pack(pady=(18,0))

    def _score_label(self, label_text, color, col):
        tk.Label(self.score_frame, text=label_text,
                 fg=color, bg=self.PANEL,
                 font=self.f_score, padx=16).grid(row=0, column=col)

    def _refresh_scores(self):
        # Rebuild score numbers dynamically
        for w in self.score_frame.winfo_children():
            w.destroy()

        col = 0
        for label, key, color in [("YOU (X)", 'X', self.ACCENT),
                                   ("DRAWS",   'draw', self.DIM),
                                   ("AI  (O)", 'O', self.ACCENT2)]:
            tk.Label(self.score_frame,
                     text=f"{label}\n{self.scores[key]}",
                     fg=color, bg=self.PANEL,
                     font=self.f_score, padx=20, justify="center"
                     ).grid(row=0, column=col)
            col += 1

    # ── window centering ───────────────────
    def center_window(self):
        self.update_idletasks()
        w = self.winfo_width()
        h = self.winfo_height()
        sw = self.winfo_screenwidth()
        sh = self.winfo_screenheight()
        self.geometry(f"+{(sw-w)//2}+{(sh-h)//2}")

    # ── game logic ─────────────────────────
    def human_move(self, idx):
        if self.game_over or not self.player_turn:
            return
        if self.board[idx] != '':
            return

        self.board[idx] = 'X'
        self.buttons[idx].config(text='×', fg=self.ACCENT)
        self.player_turn = False

        if self._check_end():
            return

        self.status_var.set("⏳  AI is thinking…")
        self.after(300, self.ai_move)

    def ai_move(self):
        move = best_move(self.board)
        if move == -1:
            return
        self.board[move] = 'O'
        self.buttons[move].config(text='○', fg=self.ACCENT2)
        self.player_turn = True

        if not self._check_end():
            self.status_var.set("▶  Your turn  (X)")

    def _check_end(self):
        result, win_cells = check_winner(self.board)
        if result is None:
            return False

        self.game_over = True
        if result == 'draw':
            self.scores['draw'] += 1
            self.status_var.set("🤝  It's a draw!")
            self._flash_all()
        elif result == 'X':
            self.scores['X'] += 1
            self.status_var.set("🎉  You win!")
            self._highlight_win(win_cells, self.WIN_HL)
        else:
            self.scores['O'] += 1
            self.status_var.set("🤖  AI wins!")
            self._highlight_win(win_cells, "#f87171")

        self._refresh_scores()
        return True

    def _highlight_win(self, cells, color):
        for i in cells:
            self.buttons[i].config(bg=color, fg=self.BG)
        self.win_cells = list(cells)

    def _flash_all(self):
        for btn in self.buttons:
            btn.config(bg=self.DIM)

    def reset_game(self):
        self.board = [''] * 9
        self.game_over = False
        self.player_turn = True
        self.win_cells = []
        for btn in self.buttons:
            btn.config(text='', bg=self.PANEL, fg=self.TEXT)
        self.status_var.set("▶  Your turn  (X)")


# ─────────────────────────────────────────
if __name__ == "__main__":
    app = TicTacToe()
    app.mainloop()